In [5]:
import pandas as pd

In [10]:
folder_path = '/Users/gtw/Desktop/新能源/电力/archive-2/PJME_hourly.csv'

In [28]:
df = pd.read_csv(folder_path, parse_dates = ['Datetime'], index_col =[0]) #把datetime转成python能识别的时间对象
df.head()
# Datetime现在变成了index，因为在下面一行，可以根据这个判断

,PJME_MW
Datetime,
2002-12-31 01:00:00,26498.0
2002-12-31 02:00:00,25147.0
2002-12-31 03:00:00,24574.0
2002-12-31 04:00:00,24393.0
2002-12-31 05:00:00,24860.0


In [25]:
df.dtypes #Datetime成功转成了时间对象, MW意思是兆瓦

Datetime    datetime64[us]
PJME_MW            float64
dtype: object

In [35]:
#Feature Extraction 
#1. 提取时间特征：进一步处理数据：年月，星期，小时
df['Year'] = df.index.year
df['Month'] = df.index.month
df['Week'] = df.index.dayofweek
df['Hour'] = df.index.hour

#Lag Features
#2. 提取滞后特征：滞后24小时
df['Lags_24hrs'] = df['PJME_MW'].shift(24)

#Scikit-learn 不认na，防止报错所以直接删掉
#3. drop
df = df.dropna()

In [37]:
df.tail() #check

,PJME_MW,Year,Month,Week,Hour,Lags_24hrs
Datetime,,,,,,
2018-01-01 20:00:00,44284.0,2018,1,0,20,45787.0
2018-01-01 21:00:00,43751.0,2018,1,0,21,45209.0
2018-01-01 22:00:00,42402.0,2018,1,0,22,43663.0
2018-01-01 23:00:00,40164.0,2018,1,0,23,41581.0
2018-01-02 00:00:00,38608.0,2018,1,1,0,39451.0


In [43]:
#Split Data
#按照时间做切分，数据从2002-2018，决定2015作为分界点
train = df.loc[df.index < '2015-01-01']
test = df.loc[df.index >= '2015-01-01']

#指定特征和目标
Features = ['Year', 'Month', 'Week', 'Hour', 'Lags_24hrs']
Target = ['PJME_MW']

#拆解X和y，完整测试集和训练集的结构
X_train = train[Features]
y_train = train[Target]

X_test = test[Features]
y_test = test[Target]

#check
print(f'训练集大小:{X_train.shape}, 测试集大小:{X_test.shape}')

训练集大小:(113902, 5), 测试集大小:(31440, 5)


In [53]:
#Training
!pip install scikit-learn

In [55]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

model = LinearRegression()
model.fit(X_train, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [56]:
#Testing
test['prediction'] = model.predict(X_test)

In [58]:
#RMSE
score = np.sqrt(mean_squared_error(y_test, test['prediction']))
print(f'测试集的误差评分：{score:0.2f}' )

测试集的误差评分：2933.87


In [62]:
#Boosting
!pip install xgboost 

  Using cached xgboost-3.2.0-py3-none-macosx_12_0_arm64.whl.metadata (2.1 kB)
Using cached xgboost-3.2.0-py3-none-macosx_12_0_arm64.whl (2.3 MB)


In [ ]:
from xgboost import XGBRegressor 